In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import numpy as np 
from typing import List

d:\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# from langchain_community.document_loaders import DirectoryLoader
# dirloc='D:\GenAI\LangChain'

# loader = DirectoryLoader(
#     dirloc,
#     glob="*.pdf",
#     loader_cls=TextLoader,
#     loader_kwargs={'encoding':'utf-8'}
# )
# documents = loader.load()

# print(f"Loaded len is {len(documents)}")
# print("First line is")
# print(documents[0].page_content[:200])

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

dirloc = r"D:\GenAI\Documents"

loader = DirectoryLoader(
    dirloc,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(f"Loaded len is {len(documents)}")
print("First line is")
print(documents[0].page_content[:200])

Loaded len is 153
First line is
AI Regulations Worldwide
ENSURING A NATIONAL POLICY FRAMEWORK FOR ARTIFICIAL
INTELLIGENCE
By the authority vested in me as President by the Constitution and the laws of the United States of
America, i


In [3]:
# print(documents[-1].page_content[:200])
documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-07T22:02:46+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-07T22:02:46+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'D:\\GenAI\\LangChain\\ai_regulation_global_report.pdf', 'total_pages': 122, 'page': 0, 'page_label': '1'}, page_content='AI Regulations Worldwide\nENSURING A NATIONAL POLICY FRAMEWORK FOR ARTIFICIAL\nINTELLIGENCE\nBy the authority vested in me as President by the Constitution and the laws of the United States of\nAmerica, it is hereby ordered:\nSection 1. Purpose. United States leadership in Artificial Intelligence (AI) will promote United States\nnational and economic security and dominance across many domains. Pursuant to Executive Order\n14179 of January 23, 2025 (Removing Barriers to American Leadership in Artificial Intelligence), I\nrevoked my predecessor’s attempt to paralyz

SPLITTING TEXT

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)
chunks=text_splitter.split_documents(documents)

print(f"Created len {len(chunks)} from len {len(documents)} documents")

Created len 1332 from len 153 documents


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)



C:\Users\frasi\AppData\Local\Temp\ipykernel_19700\3633135916.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 173.11it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
sample_text="Hello welcome to Langchain"
embedText=embeddings.embed_query(sample_text)
len(embedText)
embedText

[-0.07984120398759842,
 -0.061947014182806015,
 0.016253385692834854,
 -0.05604996532201767,
 0.010753449983894825,
 0.0009999307803809643,
 0.009470940567553043,
 0.0368102490901947,
 0.02228567562997341,
 -0.05382615700364113,
 0.02230115607380867,
 -0.030266016721725464,
 0.02280733734369278,
 0.07385393232107162,
 0.04298923537135124,
 -0.03618231788277626,
 0.013193780556321144,
 -0.06923797726631165,
 -0.03714834153652191,
 -0.011958274990320206,
 0.016782470047473907,
 -0.021757936105132103,
 -0.007223926950246096,
 -0.039880938827991486,
 0.03598407283425331,
 0.06714262068271637,
 0.06980926543474197,
 -0.030964989215135574,
 0.022152503952383995,
 -0.15335173904895782,
 0.026066143065690994,
 -0.011441570706665516,
 0.007845941931009293,
 0.004871630575507879,
 -0.03699078410863876,
 -0.01087308581918478,
 -0.011432035826146603,
 -0.018302273005247116,
 -0.03170248493552208,
 0.01670309342443943,
 0.0019648936577141285,
 -0.014816373586654663,
 0.0346425324678421,
 -0.0094205

CREATE CHROMADB VECTORE STORE

In [7]:
db_directory="./chroma_db"

vectorstore=Chroma.from_documents(
documents=chunks,
embedding=embeddings,
persist_directory=db_directory,
collection_name="rag_collection"
)

In [8]:
query="Who was the winner of India vs Pakistan in 2026 world cup?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'title': '(anonymous)', 'page': 5, 'creator': '(unspecified)', 'author': '(anonymous)', 'subject': '(unspecified)', 'trapped': '/False', 'creationdate': '2026-03-07T23:37:37+05:00', 'moddate': '2026-03-07T23:37:37+05:00', 'page_label': '6', 'producer': 'ReportLab PDF Library - (opensource)', 'keywords': '', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'total_pages': 31}, page_content="India vs Pakistan | ICC Men's T20 World Cup 2024 | Match\nHighlights | BCCI.tv"),
 Document(metadata={'creationdate': '2026-03-07T23:37:37+05:00', 'trapped': '/False', 'subject': '(unspecified)', 'keywords': '', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'title': '(anonymous)', 'creator': '(unspecified)', 'total_pages': 31, 'producer': 'ReportLab PDF Library - (opensource)', 'page_label': '6', 'moddate': '2026-03-07T23:37:37+05:00', 'author': '(anonymous)', 'page': 5}, page_content="India vs Pakistan | ICC Men's T20 World Cup 2024 | M

In [9]:
query="How much did Suryakumar Yadav score in India vs Pakistan in 2026 world cup?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'page': 5, 'page_label': '6', 'subject': '(unspecified)', 'title': '(anonymous)', 'total_pages': 31, 'creator': '(unspecified)', 'trapped': '/False', 'moddate': '2026-03-07T23:37:37+05:00', 'creationdate': '2026-03-07T23:37:37+05:00', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'keywords': '', 'author': '(anonymous)'}, page_content="India vs Pakistan | ICC Men's T20 World Cup 2024 | Match\nHighlights | BCCI.tv"),
 Document(metadata={'title': '(anonymous)', 'trapped': '/False', 'keywords': '', 'page_label': '6', 'creator': '(unspecified)', 'page': 5, 'producer': 'ReportLab PDF Library - (opensource)', 'moddate': '2026-03-07T23:37:37+05:00', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'author': '(anonymous)', 'subject': '(unspecified)', 'total_pages': 31, 'creationdate': '2026-03-07T23:37:37+05:00'}, page_content="India vs Pakistan | ICC Men's T20 World Cup 2024 | M

In [10]:
print(query)
for i,doc in enumerate(similar_docs):
    print(i+1)
    print(doc.page_content[:200])
    print(doc.metadata.get('source','Unknown'))

How much did Suryakumar Yadav score in India vs Pakistan in 2026 world cup?
1
India vs Pakistan | ICC Men's T20 World Cup 2024 | Match
Highlights | BCCI.tv
D:\GenAI\LangChain\cricket_worldcup_match_reports.pdf
2
India vs Pakistan | ICC Men's T20 World Cup 2024 | Match
Highlights | BCCI.tv
D:\GenAI\LangChain\cricket_worldcup_match_reports.pdf
3
Shivam Dube’s explosive 66 off 31 balls anchored India’s 193/6, with contributions from Tilak
Varma, Suryakumar Yadav, and Hardik Pandya. The Dutch fought hard, led by Bas de Leede’s 33,
but Varun Cha
D:\GenAI\LangChain\cricket_worldcup_match_reports.pdf


SIMILARTY SEARCH WITH SCORES

In [11]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'page_label': '6', 'total_pages': 31, 'title': '(anonymous)', 'creationdate': '2026-03-07T23:37:37+05:00', 'moddate': '2026-03-07T23:37:37+05:00', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'trapped': '/False', 'author': '(anonymous)', 'subject': '(unspecified)', 'page': 5, 'keywords': '', 'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)'}, page_content="India vs Pakistan | ICC Men's T20 World Cup 2024 | Match\nHighlights | BCCI.tv"),
  0.48602741956710815),
 (Document(metadata={'title': '(anonymous)', 'source': 'D:\\GenAI\\LangChain\\cricket_worldcup_match_reports.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'author': '(anonymous)', 'moddate': '2026-03-07T23:37:37+05:00', 'trapped': '/False', 'creationdate': '2026-03-07T23:37:37+05:00', 'page_label': '6', 'creator': '(unspecified)', 'keywords': '', 'subject': '(unspecified)', 'total_pages': 31, 'page': 5}, page_content="India vs Pakistan | ICC Me

INITIALZING RAG

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:0.5b",  # Use the exact tag you pulled
    temperature=0.1,
    num_ctx=2048
)

# llm = ChatOpenAI(
#     model="qwen2.5:0.5b",
#     # Point to your local Ollama server's OpenAI-compatible endpoint
#     base_url="http://localhost:11434/v1",
#     # Ollama doesn't check the key, but the SDK requires one to be present
#     api_key="ollama" 
# )


In [16]:
response = llm.invoke("Hello!")
print(response.content)

Hello! How can I assist you today?


In [48]:
from langchain.chat_models.base import init_chat_model

llm=init_chat_model("ollama:qwen2.5:0.5b",num_ctx=2048)
llm

ChatOllama(model='qwen2.5:0.5b', num_ctx=2048)

In [18]:
llm.invoke("What is AI")

AIMessage(content='AI stands for "Artificial Intelligence," which is a subset of artificial intelligence and machine learning. It refers to the development of computer systems that can perform tasks that typically require human intelligence, such as image recognition, language understanding, decision making, and natural language processing. AI is becoming increasingly prevalent in various fields, including healthcare, finance, transportation, and gaming. AI has the potential to transform many industries by enabling machines to learn and make decisions without human intervention.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:0.5b', 'created_at': '2026-03-08T07:49:29.071213Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3912134200, 'load_duration': 2988718200, 'prompt_eval_count': 32, 'prompt_eval_duration': 46355900, 'eval_count': 93, 'eval_duration': 799441900, 'logprobs': None, 'model_name': 'qwen2.5:0.5b', 'model_provider': 'ollama'}, id='lc_run--019ccc6c-24

RAG CHAIN

In [23]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [24]:
### Make the Vector store to retriver
retriever =vectorstore.as_retriever(
    search_kwarg={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DF2806F350>, search_kwargs={})

In [38]:
### Prompt Template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
system_prompt=""" You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answert, Just say that you don't know.
Use three sentences maximum and keep the answer concise 

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

chain = prompt | llm | StrOutputParser()


In [40]:
chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answert, Just say that you don't know.\nUse three sentences maximum and keep the answer concise \n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(model='qwen2.5:0.5b')
| StrOutputParser()

DOC CHAIN CREATION

In [41]:
# # from langchain_classic.combine_documents import create_stuff_documents_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# document_chain=create_stuff_documents_chain(llm,prompt)
# document_chain

In [42]:
rag_chain=create_retrieval_chain(retriever,chain)

In [43]:
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DF2806F350>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answert, Just say that you don't know.\nUse three sentences maximum and keep the answer concise \n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_vari

In [45]:
response=rag_chain.invoke({"input":"What is the purpose of AI Regulations Worldwide"})

In [47]:
response

{'input': 'What is the purpose of AI Regulations Worldwide',
 'context': [Document(metadata={'creationdate': '2026-03-07T22:02:46+05:00', 'title': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'page_label': '82', 'producer': 'ReportLab PDF Library - (opensource)', 'page': 81, 'creator': '(unspecified)', 'trapped': '/False', 'moddate': '2026-03-07T22:02:46+05:00', 'total_pages': 122, 'source': 'D:\\GenAI\\LangChain\\ai_regulation_global_report.pdf', 'author': '(anonymous)'}, page_content='to have a clear collective vision for how AI should be regulated. This has\nresulted in divergent approaches around the world—from comprehensive legislation in the\nEuropean Union (EU) to technology-specific rules in China, and voluntary commitments in the\nUnited States. Despite these differences, global policymakers seem to agree on one thing—we\nmust leverage the power of AI while mitigating its risks.\nWhere does this leave India on AI regulation? The existing body of literature concer

In [46]:
response['answer']

'The purpose of AI Regulations Worldwide is to establish clear collective vision for how AI should be regulated. This has resulted in divergent approaches around the world, with some countries adopting comprehensive legislation while others are implementing technology-specific rules or voluntary commitments. Despite these differences, global policymakers seem to agree on the need to leverage the power of AI while mitigating its risks.'